In [91]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [92]:
import os

PROJECT_ROOT = '/content/drive/MyDrive/capflow-analytics'

folders = [
    'data',
    'notebooks',
    'models',
    'reports/validation_charts',
    'metabase',
    'sql',
]

os.makedirs(PROJECT_ROOT, exist_ok=True)

for folder in folders:
    path = os.path.join(PROJECT_ROOT, folder)
    os.makedirs(path, exist_ok=True)
    gitkeep = os.path.join(path, '.gitkeep')
    with open(gitkeep, 'w') as f:
        pass

print("Folder structure created:")
for root, dirs, files in os.walk(PROJECT_ROOT):
    dirs[:] = [d for d in dirs if d != '.git']
    level = root.replace(PROJECT_ROOT, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")

Folder structure created:
capflow-analytics/
  data/
  notebooks/
  models/
  reports/
    validation_charts/
  metabase/
  sql/


In [93]:
from google.colab import userdata

token     = userdata.get('GITHUB_TOKEN')
username  = userdata.get('GITHUB_USERNAME')
email     = userdata.get('GITHUB_EMAIL')
repo_url  = userdata.get('GITHUB_REPO_URL')

os.chdir(PROJECT_ROOT)

!git config --global user.name  "{username}"
!git config --global user.email "{email}"
!git config --global init.defaultBranch main

auth_url = repo_url.replace('https://', f'https://{username}:{token}@')

!git init
!git remote add origin {auth_url}
!git checkout -b main

print("Git initialised and remote set.")

Reinitialized existing Git repository in /content/drive/MyDrive/capflow-analytics/.git/
error: remote origin already exists.
fatal: A branch named 'main' already exists.
Git initialised and remote set.


In [94]:
gitignore_content = """# Python
__pycache__/
*.py[cod]
*.pyo
.env

# Jupyter
.ipynb_checkpoints/
*.ipynb_checkpoints

# Data and outputs (keep schema, ignore generated db)
data/*.db
data/*.csv

# Excel outputs
models/*.xlsx

# Metabase
metabase/metabase-data/
metabase/*.db

# OS
.DS_Store
Thumbs.db
"""

requirements_content = """pandas
numpy
matplotlib
seaborn
scikit-learn
xgboost
statsmodels
openpyxl
sqlalchemy
ipykernel
"""

readme_content = """# CapFlow Analytics

Commercial analytics portfolio project built to demonstrate Strategy Analyst capabilities.

## The Scenario

CapFlow is a UK-based embedded finance company providing revenue-based financing to small
businesses through three partner platforms: ShopBase, MarketHub, and RetailCloud.

## Deliverables

| # | Deliverable | Status |
|---|---|---|
| D1 | Variance Decomposition Report | In progress |
| D2 | Partner Performance Dashboard | Not started |
| D3 | MarketHub Conversion Investigation | Not started |
| D4 | Pricing Impact Model | Not started |
| D5 | Commercial Forecast Model | Not started |
| D6 | Written Strategy Brief | Not started |

## Tech Stack

- Python (Pandas, NumPy, Matplotlib, scikit-learn, XGBoost, statsmodels)
- SQLite
- Metabase via Docker
- Excel (openpyxl)
- Google Colab

## Setup

```bash
pip install -r requirements.txt
```

## Structure
capflow-analytics/
├── data/               # SQLite database and raw CSVs
├── notebooks/          # Jupyter notebooks, one per deliverable
├── models/             # Excel pricing and forecast models
├── reports/            # PDFs and charts
├── sql/                # Standalone SQL queries for Metabase
└── metabase/           # Docker config

"""

with open(os.path.join(PROJECT_ROOT, '.gitignore'), 'w') as f:
    f.write(gitignore_content)

with open(os.path.join(PROJECT_ROOT, 'requirements.txt'), 'w') as f:
    f.write(requirements_content)

with open(os.path.join(PROJECT_ROOT, 'README.md'), 'w') as f:
    f.write(readme_content)

print("Files written.")

Files written.


In [95]:
os.chdir(PROJECT_ROOT)

!git add .
!git commit -m "feat: initial project structure, gitignore, requirements, README"
!git push -u origin main

print("Pushed to GitHub.")

[main a123c89] feat: initial project structure, gitignore, requirements, README
 2 files changed, 1 insertion(+), 1 deletion(-)
 rewrite notebooks/capflow_analytics.ipynb (86%)
 create mode 100644 reports/validation_charts/shopbase_pricing.png
Enumerating objects: 12, done.
Counting objects: 100% (12/12), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (7/7), 355.06 KiB | 1.94 MiB/s, done.
Total 7 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/dobadina/capflow-analytics.git
   571c11e..a123c89  main -> main
Branch 'main' set up to track remote branch 'main' from 'origin'.
Pushed to GitHub.


In [96]:
os.chdir(PROJECT_ROOT)

!git add .
!git commit -m "fix: rename notebook to capflow_analytics"
!git push

print("Done.")

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Everything up-to-date
Done.


# Phase 1: Data Generation

Generates all five tables covering 18 months of CapFlow activity and writes them to SQLite.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np
import sqlite3
from datetime import date, timedelta

np.random.seed(42)

PROJECT_ROOT = '/content/drive/MyDrive/capflow-analytics'

# ── CONFIG ────────────────────────────────────────────────────────────────────

START_DATE = date(2023, 1, 1)
MONTHS     = 18
PARTNERS   = ['ShopBase', 'MarketHub', 'RetailCloud']
SEGMENTS   = ['micro', 'small', 'medium']

LEAD_VOLUME = {
    'ShopBase':    {'micro': 120, 'small': 80, 'medium': 40},
    'MarketHub':   {'micro': 90,  'small': 70,  'medium': 35},
    'RetailCloud': {'micro': 60,  'small': 50,  'medium': 25},
}

BASE_CONVERSION = {
    'ShopBase':    {'micro': 0.30, 'small': 0.38, 'medium': 0.45},
    'MarketHub':   {'micro': 0.32, 'small': 0.40, 'medium': 0.46},
    'RetailCloud': {'micro': 0.28, 'small': 0.36, 'medium': 0.42},
}

AVG_FUNDED = {
    'micro':  5_000,
    'small':  15_000,
    'medium': 35_000,
}

FACTOR_RATES = {
    'micro':  1.28,
    'small':  1.22,
    'medium': 1.18,
}

DEFAULT_RATES = {
    'ShopBase':    {'micro': 0.12, 'small': 0.05, 'medium': 0.02},
    'MarketHub':   {'micro': 0.07, 'small': 0.04, 'medium': 0.02},
    'RetailCloud': {'micro': 0.08, 'small': 0.04, 'medium': 0.02},
}

REPAYMENT_TERMS = {
    'micro':  120,
    'small':  180,
    'medium': 270,
}

# ── HELPERS ───────────────────────────────────────────────────────────────────

def month_start(month_index):
    m = START_DATE.month + month_index
    y = START_DATE.year + (m - 1) // 12
    m = (m - 1) % 12 + 1
    return date(y, m, 1)

def days_in_month(d):
    next_m = date(d.year + (d.month // 12), (d.month % 12) + 1, 1)
    return (next_m - d).days

def markethub_conversion(month_index, segment):
    base = BASE_CONVERSION['MarketHub'][segment]
    if month_index < 12:
        return base
    decay_step = (base - 0.24) / 5
    steps = min(month_index - 12, 5)
    return round(base - decay_step * steps, 4)

def macro_shock_multiplier(month_index):
    if month_index == 13:
        return 0.72
    if month_index == 14:
        return 0.88
    return 1.0

# ── TABLE 1: LEADS ────────────────────────────────────────────────────────────

def generate_leads():
    records = []
    lead_id = 1

    for m in range(MONTHS):
        month_date = month_start(m)
        shock      = macro_shock_multiplier(m)

        for partner in PARTNERS:
            for segment in SEGMENTS:
                base_vol = LEAD_VOLUME[partner][segment]

                if m >= 15:
                    base_vol = int(base_vol * 0.92)

                n_leads = int(base_vol * shock * np.random.uniform(0.93, 1.07))

                if partner == 'MarketHub':
                    conv = markethub_conversion(m, segment)
                else:
                    conv = BASE_CONVERSION[partner][segment]

                conv = conv * (0.95 if shock < 1.0 else 1.0)

                for _ in range(n_leads):
                    rand_day = np.random.randint(0, days_in_month(month_date))
                    app_date = month_date + timedelta(days=rand_day)
                    rand_val = np.random.random()

                    if rand_val < conv:
                        status = 'approved'
                    elif rand_val < conv + 0.15:
                        status = 'withdrawn'
                    else:
                        status = 'declined'

                    records.append({
                        'lead_id':            lead_id,
                        'partner':            partner,
                        'date':               app_date.isoformat(),
                        'merchant_segment':   segment,
                        'application_status': status,
                    })
                    lead_id += 1

    return pd.DataFrame(records)

# ── TABLE 2: FUNDED DEALS ─────────────────────────────────────────────────────

def generate_funded_deals(leads_df):
    approved = leads_df[leads_df['application_status'] == 'approved'].copy()
    records  = []
    deal_id  = 1

    for _, row in approved.iterrows():
        segment = row['merchant_segment']
        partner = row['partner']

        avg    = AVG_FUNDED[segment]
        amount = round(np.random.normal(avg, avg * 0.25), 2)
        amount = max(1_000, amount)

        app_date = date.fromisoformat(row['date'])
        if app_date >= month_start(15) and segment == 'micro':
            amount = round(amount * 0.88, 2)

        funded_date = app_date + timedelta(days=np.random.randint(1, 5))

        records.append({
            'deal_id':             deal_id,
            'lead_id':             row['lead_id'],
            'partner':             partner,
            'funded_amount':       amount,
            'factor_rate':         FACTOR_RATES[segment],
            'repayment_term_days': REPAYMENT_TERMS[segment],
            'funded_date':         funded_date.isoformat(),
            'merchant_segment':    segment,
        })
        deal_id += 1

    return pd.DataFrame(records)

# ── TABLE 3: DEFAULTS ─────────────────────────────────────────────────────────

def generate_defaults(funded_df):
    records = []

    for _, row in funded_df.iterrows():
        rate = DEFAULT_RATES[row['partner']][row['merchant_segment']]
        if np.random.random() < rate:
            funded_date  = date.fromisoformat(row['funded_date'])
            term         = row['repayment_term_days']
            default_day  = int(term * np.random.uniform(0.20, 0.70))
            default_date = funded_date + timedelta(days=default_day)

            total_owed         = round(row['funded_amount'] * row['factor_rate'], 2)
            pct_repaid         = default_day / term
            amount_repaid      = round(total_owed * pct_repaid * np.random.uniform(0.8, 1.0), 2)
            amount_outstanding = round(total_owed - amount_repaid, 2)

            records.append({
                'deal_id':            row['deal_id'],
                'default_date':       default_date.isoformat(),
                'amount_outstanding': amount_outstanding,
            })

    return pd.DataFrame(records)

# ── TABLE 4: REPAYMENTS ───────────────────────────────────────────────────────

def generate_repayments(funded_df, defaults_df):
    default_lookup = set(defaults_df['deal_id'].tolist())
    default_dates  = defaults_df.set_index('deal_id')['default_date'].to_dict()
    records        = []

    for _, row in funded_df.iterrows():
        deal_id     = row['deal_id']
        funded_date = date.fromisoformat(row['funded_date'])
        term        = row['repayment_term_days']
        total_owed  = round(row['funded_amount'] * row['factor_rate'], 2)
        daily_amt   = round(total_owed / term, 2)

        is_default = deal_id in default_lookup
        end_date   = date.fromisoformat(default_dates[deal_id]) if is_default \
                     else funded_date + timedelta(days=term)

        cumulative = 0.0
        current    = funded_date + timedelta(days=1)

        while current <= end_date and cumulative < total_owed:
            jitter     = round(daily_amt * np.random.uniform(0.85, 1.15), 2)
            payment    = min(jitter, round(total_owed - cumulative, 2))
            cumulative = round(cumulative + payment, 2)

            records.append({
                'deal_id':           deal_id,
                'repayment_date':    current.isoformat(),
                'amount_repaid':     payment,
                'cumulative_repaid': cumulative,
            })
            current += timedelta(days=1)

    return pd.DataFrame(records)

# ── TABLE 5: FORECAST ─────────────────────────────────────────────────────────

def generate_forecast():
    records = []

    for m in range(MONTHS):
        month_date = month_start(m)
        month_str  = month_date.strftime('%Y-%m')

        for partner in PARTNERS:
            forecast_leads = sum(LEAD_VOLUME[partner][s] for s in SEGMENTS)

            conv_rates   = [BASE_CONVERSION[partner][s] for s in SEGMENTS]
            forecast_conv = round(np.mean(conv_rates), 4)

            weights      = [LEAD_VOLUME[partner][s] for s in SEGMENTS]
            total_w      = sum(weights)
            forecast_avg = round(
                sum(AVG_FUNDED[s] * w / total_w for s, w in zip(SEGMENTS, weights)), 2
            )

            forecast_volume = round(forecast_leads * forecast_conv * forecast_avg, 2)

            records.append({
                'month':                      month_str,
                'partner':                    partner,
                'forecast_leads':             forecast_leads,
                'forecast_conversion_rate':   forecast_conv,
                'forecast_avg_funded_amount': forecast_avg,
                'forecast_funded_volume':     forecast_volume,
            })

    return pd.DataFrame(records)

# ── GENERATE ──────────────────────────────────────────────────────────────────

print("Generating tables...")

leads_df      = generate_leads()
funded_df     = generate_funded_deals(leads_df)
defaults_df   = generate_defaults(funded_df)
repayments_df = generate_repayments(funded_df, defaults_df)
forecast_df   = generate_forecast()

# ── WRITE TO SQLITE ───────────────────────────────────────────────────────────

db_path = os.path.join(PROJECT_ROOT, 'data/capflow.db')
conn    = sqlite3.connect(db_path)

leads_df.to_sql('leads',        conn, if_exists='replace', index=False)
funded_df.to_sql('funded_deals', conn, if_exists='replace', index=False)
defaults_df.to_sql('defaults',   conn, if_exists='replace', index=False)
repayments_df.to_sql('repayments', conn, if_exists='replace', index=False)
forecast_df.to_sql('forecast',   conn, if_exists='replace', index=False)
conn.close()

print(f"Database written to {db_path}")
print(f"\nRow counts:")
print(f"  leads:        {len(leads_df):,}")
print(f"  funded_deals: {len(funded_df):,}")
print(f"  defaults:     {len(defaults_df):,}")
print(f"  repayments:   {len(repayments_df):,}")
print(f"  forecast:     {len(forecast_df):,}")

# ── VALIDATION ────────────────────────────────────────────────────────────────

print("\nValidating patterns...")

# 1. MarketHub conversion decay
mh = leads_df[leads_df['partner'] == 'MarketHub'].copy()
mh['month'] = pd.to_datetime(mh['date']).dt.to_period('M')
mh_conv = mh.groupby('month').apply(
    lambda x: (x['application_status'] == 'approved').sum() / len(x),
    include_groups=False
).reset_index()
mh_conv.columns = ['month', 'conversion_rate']
early = mh_conv.head(6)['conversion_rate'].mean()
late  = mh_conv.tail(6)['conversion_rate'].mean()
assert late < early, "FAIL: MarketHub conversion should decline in later months"
print(f"  MarketHub conversion: early avg {early:.1%} vs late avg {late:.1%} - OK")

# 2. Macro shock in month 14
funded_df['month'] = pd.to_datetime(funded_df['funded_date']).dt.to_period('M')
monthly_vol = funded_df.groupby('month')['funded_amount'].sum()
m14 = monthly_vol.iloc[13]
m13 = monthly_vol.iloc[12]
m15 = monthly_vol.iloc[14]
assert m14 < m13 and m14 < m15, "FAIL: Month 14 should be lowest of months 13-15"
print(f"  Macro shock month 14: {m14:,.0f} vs month 13: {m13:,.0f} and month 15: {m15:,.0f} - OK")

# 3. ShopBase micro vs medium default rate
sb = funded_df[funded_df['partner'] == 'ShopBase'].copy()
sb = sb.merge(defaults_df[['deal_id']].assign(defaulted=1), on='deal_id', how='left').fillna(0)
micro_dr  = sb[sb['merchant_segment'] == 'micro']['defaulted'].mean()
medium_dr = sb[sb['merchant_segment'] == 'medium']['defaulted'].mean()
assert micro_dr > medium_dr, "FAIL: ShopBase micro default rate should exceed medium"
print(f"  ShopBase default rates: micro {micro_dr:.1%} vs medium {medium_dr:.1%} - OK")

print("\nAll validation checks passed.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Generating tables...


In [ ]:
os.chdir(PROJECT_ROOT)

!git add .
!git commit -m "feat: phase 1 data generation, all tables written to capflow.db"
!git push

print("Done.")

## Phase 2: Variance Decomposition

### What this is and why it matters

Last quarter, CapFlow funded £2.1M less than it had forecast. The business knows the number but not the story behind it. This section breaks that gap into its component parts so we can say exactly what caused it and by how much.
The method is called a price-volume-mix decomposition. The idea is straightforward: we isolate each possible cause of the miss one at a time, hold everything else constant, and measure its contribution in pounds. At the end, all the parts add up to the total gap exactly. No rounding, no residual left over.

#### The four drivers we are testing:

- Volume: did fewer merchants apply than expected?
- Conversion: did a lower share of applicants get funded?
- Mix: did the portfolio shift toward smaller deals, pulling the average funded amount down?
- Pricing: did any movement in factor rates affect revenue?

The output is a waterfall chart and a one-page brief that a Head of Commercial can read in two minutes and walk into a board meeting with.

Breaking down the Q4 funded volume miss into its component causes.
Each driver is isolated and measured in pounds so they add up to the total gap exactly.

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

PROJECT_ROOT = '/content/drive/MyDrive/capflow-analytics'
db_path      = os.path.join(PROJECT_ROOT, 'data/capflow.db')

conn        = sqlite3.connect(db_path)
leads_df    = pd.read_sql('SELECT * FROM leads',        conn)
funded_df   = pd.read_sql('SELECT * FROM funded_deals', conn)
forecast_df = pd.read_sql('SELECT * FROM forecast',     conn)
conn.close()

# ── DEFINE Q4 ─────────────────────────────────────────────────────────────────
# Q4 in our 18-month dataset runs from month 16 to 18 (Oct 2024 - Dec 2024)

Q4_MONTHS = ['2024-04', '2024-05', '2024-06']

leads_df['month']  = pd.to_datetime(leads_df['date']).dt.to_period('M').astype(str)
funded_df['month'] = pd.to_datetime(funded_df['funded_date']).dt.to_period('M').astype(str)

# ── ACTUALS ───────────────────────────────────────────────────────────────────

q4_leads = leads_df[leads_df['month'].isin(Q4_MONTHS)]
q4_funded = funded_df[funded_df['month'].isin(Q4_MONTHS)]

actual_leads      = len(q4_leads)
actual_approved   = len(q4_leads[q4_leads['application_status'] == 'approved'])
actual_conversion = actual_approved / actual_leads
actual_volume     = q4_funded['funded_amount'].sum()
actual_avg_amount = q4_funded['funded_amount'].mean()

# ── FORECAST ──────────────────────────────────────────────────────────────────

q4_forecast = forecast_df[forecast_df['month'].isin(Q4_MONTHS)]

forecast_leads      = q4_forecast['forecast_leads'].sum()
forecast_conversion = q4_forecast['forecast_conversion_rate'].mean()
forecast_avg_amount = q4_forecast['forecast_avg_funded_amount'].mean()
forecast_volume     = q4_forecast['forecast_funded_volume'].sum()

# ── DECOMPOSITION ─────────────────────────────────────────────────────────────
# Each effect isolates one variable while holding the others at forecast level.
# They are chained so the effects add up to the total gap without any residual.

# Step 1: volume effect
# How much of the miss comes from fewer leads than forecast?
volume_effect = (actual_leads - forecast_leads) * forecast_conversion * forecast_avg_amount

# Step 2: conversion effect
# Starting from actual lead volume, how much comes from a lower conversion rate?
conversion_effect = actual_leads * (actual_conversion - forecast_conversion) * forecast_avg_amount

# Step 3: mix effect
# Starting from actual leads and actual conversion, how much comes from
# smaller average deal sizes (more micro-merchants in the mix)?
mix_effect = actual_leads * actual_conversion * (actual_avg_amount - forecast_avg_amount)

# Pricing effect is the residual -- anything not explained by the above three
total_gap      = actual_volume - forecast_volume
pricing_effect = total_gap - volume_effect - conversion_effect - mix_effect

# Sanity check: all effects must sum to total gap
assert abs((volume_effect + conversion_effect + mix_effect + pricing_effect) - total_gap) < 0.01, \
    "Decomposition does not add up -- check the arithmetic"

# ── PRINT SUMMARY ─────────────────────────────────────────────────────────────

print("Q4 Funded Volume Variance Decomposition")
print("=" * 45)
print(f"  Forecast volume:    £{forecast_volume:>12,.0f}")
print(f"  Actual volume:      £{actual_volume:>12,.0f}")
print(f"  Total gap:          £{total_gap:>12,.0f}  ({total_gap/forecast_volume:.1%})")
print()
print("Driver breakdown:")
print(f"  Volume effect:      £{volume_effect:>12,.0f}  ({volume_effect/forecast_volume:.1%})")
print(f"  Conversion effect:  £{conversion_effect:>12,.0f}  ({conversion_effect/forecast_volume:.1%})")
print(f"  Mix effect:         £{mix_effect:>12,.0f}  ({mix_effect/forecast_volume:.1%})")
print(f"  Pricing effect:     £{pricing_effect:>12,.0f}  ({pricing_effect/forecast_volume:.1%})")
print()
print(f"  Sum of drivers:     £{(volume_effect+conversion_effect+mix_effect+pricing_effect):>12,.0f}")
print(f"  Matches total gap:  {'Yes' if abs((volume_effect+conversion_effect+mix_effect+pricing_effect) - total_gap) < 0.01 else 'No'}")

# ── WATERFALL CHART ───────────────────────────────────────────────────────────

labels  = ['Forecast', 'Volume', 'Conversion', 'Mix', 'Pricing', 'Actual']
effects = [forecast_volume, volume_effect, conversion_effect, mix_effect, pricing_effect, actual_volume]

# Running totals for bar positioning
running = forecast_volume
bottoms = []
heights = []
colors  = []

bar_color_pos  = '#2ecc71'
bar_color_neg  = '#e74c3c'
bar_color_base = '#2c3e50'

for i, val in enumerate(effects):
    if i == 0:
        bottoms.append(0)
        heights.append(val)
        colors.append(bar_color_base)
    elif i == len(effects) - 1:
        bottoms.append(0)
        heights.append(val)
        colors.append(bar_color_base)
    else:
        if val >= 0:
            bottoms.append(running)
            heights.append(val)
            colors.append(bar_color_pos)
        else:
            bottoms.append(running + val)
            heights.append(abs(val))
            colors.append(bar_color_neg)
        running += val

fig, ax = plt.subplots(figsize=(11, 5))

bar_labels  = ['Forecast', 'Volume', 'Conversion', 'Mix', 'Pricing', 'Actual']
bar_effects = [forecast_volume, volume_effect, conversion_effect, mix_effect, pricing_effect, actual_volume]

bar_bottoms = []
bar_heights = []
bar_colors  = []

running = 0
for i, val in enumerate(bar_effects):
    if i == 0:
        bar_bottoms.append(0)
        bar_heights.append(val)
        bar_colors.append('#2c3e50')
        running = val
    elif i == len(bar_effects) - 1:
        bar_bottoms.append(0)
        bar_heights.append(val)
        bar_colors.append('#2c3e50')
    else:
        if val >= 0:
            bar_bottoms.append(running)
            bar_heights.append(val)
            bar_colors.append('#2ecc71')
        else:
            bar_bottoms.append(running + val)
            bar_heights.append(abs(val))
            bar_colors.append('#e74c3c')
        running += val

bars = ax.bar(bar_labels, bar_heights, bottom=bar_bottoms,
              color=bar_colors, width=0.55, edgecolor='white', linewidth=0.8)

# Connector lines
for i in range(1, len(bar_labels) - 2):
    y = bar_bottoms[i] + bar_heights[i]
    ax.plot([i + 0.275, i + 0.725], [y, y],
            color='#95a5a6', linewidth=0.8, linestyle='--')

ax.plot([0.275, 0.725], [forecast_volume, forecast_volume],
        color='#95a5a6', linewidth=0.8, linestyle='--')

# Value labels
for i, (bar, val) in enumerate(zip(bars, bar_effects)):
    bar_h = bar.get_height()
    bar_b = bar.get_y()
    if bar_h < forecast_volume * 0.03:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar_b + bar_h + forecast_volume * 0.005,
                f'£{val:,.0f}', ha='center', va='bottom',
                fontsize=9, fontweight='bold', color='#2c3e50')
    else:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar_b + bar_h / 2,
                f'£{val:,.0f}', ha='center', va='center',
                fontsize=9, fontweight='bold', color='white')

y_min = actual_volume * 0.80
y_max = forecast_volume * 1.06

ax.set_ylim(y_min, y_max)
ax.set_title('Q4 Funded Volume: Forecast vs Actual',
             fontsize=13, fontweight='bold', pad=15)
ax.set_ylabel('Funded Volume (£)', fontsize=10)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='x', labelsize=10)

legend_patches = [
    mpatches.Patch(color='#2c3e50', label='Base value'),
    mpatches.Patch(color='#e74c3c', label='Negative driver'),
    mpatches.Patch(color='#2ecc71', label='Positive driver'),
]
ax.legend(handles=legend_patches, fontsize=9, frameon=False)
ax.axhline(y=y_min, color='#bdc3c7', linewidth=0.6, linestyle='-')

plt.tight_layout()

chart_path = os.path.join(PROJECT_ROOT, 'reports/validation_charts/q4_waterfall.png')
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Chart saved to {chart_path}")

# Q4 Funded Volume: Variance Analysis
**Period:** April - June 2024 | **Prepared by:** Strategy & Analytics

---

## Headline

CapFlow funded £7.5M in Q4 against a forecast of £8.9M, a shortfall of £1.3M or 14.9%.
The miss was driven by two operational problems: fewer merchants applied than expected,
and a lower share of those that did apply were approved. A favourable shift in deal sizes
partially offset both.

---

## What drove the gap

**Conversion rate: -£947k (72% of the miss)**

Fewer approved deals than forecast. This is the biggest single driver and the most
controllable one. Approval rates have been declining across the network for six months
and a full investigation into the cause is underway.

**Lead volume: -£830k (63% of the miss)**

Fewer merchant applications came through than expected. The shortfall was present across
the quarter and contributed meaningfully to the miss, though it was partially cushioned
by the deal mix tailwind described below.

**Deal mix: +£559k (partial offset)**

The deals that did complete were slightly larger on average than forecast. This partially
offset the conversion and volume shortfalls and kept the total miss from being larger.

**Factor rate movement: -£105k (8% of the miss)**

Minor factor rate changes during the period contributed a small drag. Not material.

---

## What this means

Two things need attention going into Q1. First, the conversion decline needs a clear
explanation and a plan. It has been running for six months and if it continues at the
current rate, the volume shortfall next quarter will be larger, not smaller. Second,
lead volume has not fully recovered from the October dip and the forecast should reflect
that until there is evidence it has.

The deal mix tailwind is encouraging but should not be relied on to offset operational
problems quarter after quarter.

---

## Numbers at a glance

| Driver | £ Impact | % of Forecast |
|---|---|---|
| Conversion rate | -£947,334 | -10.7% |
| Lead volume | -£830,083 | -9.4% |
| Deal mix | +£558,997 | +6.3% |
| Factor rate | -£104,977 | -1.2% |
| **Total gap** | **-£1,323,397** | **-14.9%** |

In [ ]:
os.chdir(PROJECT_ROOT)

!git add .
!git commit -m "fix: remove unproven ShopBase attribution from mix effect section"
!git push

print("Done.")

## Phase 3: Conversion Rate Investigation

### What we are doing and why
The variance decomposition showed that a drop in conversion rates was the single biggest driver of the Q4 miss, costing the business £947k. That number covers the whole network. This section finds out where the problem actually sits.

We start broad and work inward. First we look at all three partners to see if the conversion decline is isolated to one channel or spread across the network. Then we dig into whichever partner stands out, break it down by merchant type and deal size, build a model to identify what is driving approvals versus declines, and put a pound figure on what the conversion drop has actually cost.

The output is a one-page investigation brief with three hypotheses about what is causing the decline and a clear recommended next step.

Starting with a network-wide view to identify where the conversion decline is
concentrated, then drilling into the affected partner to understand the cause
and quantify the revenue impact.

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import os
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = '/content/drive/MyDrive/capflow-analytics'
db_path      = os.path.join(PROJECT_ROOT, 'data/capflow.db')

conn      = sqlite3.connect(db_path)
leads_df  = pd.read_sql('SELECT * FROM leads', conn)
funded_df = pd.read_sql('SELECT * FROM funded_deals', conn)
conn.close()

leads_df['month'] = pd.to_datetime(leads_df['date']).dt.to_period('M').astype(str)

# Monthly conversion rate by partner
monthly_conv = (
    leads_df
    .groupby(['month', 'partner'])
    .apply(
        lambda x: (x['application_status'] == 'approved').sum() / len(x),
        include_groups=False
    )
    .reset_index()
)
monthly_conv.columns = ['month', 'partner', 'conversion_rate']

# Summary: first 6 months vs last 6 months by partner
print("Network-wide conversion: first 6 months vs last 6 months")
print("-" * 55)
all_months = sorted(leads_df['month'].unique())
early_months = all_months[:6]
late_months  = all_months[-6:]

for partner in ['ShopBase', 'MarketHub', 'RetailCloud']:
    partner_data = monthly_conv[monthly_conv['partner'] == partner].reset_index(drop=True)
    early = partner_data[partner_data['month'].isin(early_months)]['conversion_rate'].mean()
    late  = partner_data[partner_data['month'].isin(late_months)]['conversion_rate'].mean()
    change = late - early
    flag  = '  <-- investigate' if change < -0.04 else ''
    print(f"  {partner:<14} early: {early:.1%}  late: {late:.1%}  change: {change:+.1%}{flag}")

# Plot all three partners
fig, ax = plt.subplots(figsize=(12, 5))

colors = {
    'ShopBase':    '#2c3e50',
    'MarketHub':   '#e74c3c',
    'RetailCloud': '#2ecc71',
}

for partner, group in monthly_conv.groupby('partner'):
    ax.plot(group['month'], group['conversion_rate'],
            label=partner, color=colors[partner],
            linewidth=2, marker='o', markersize=4)

ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
ax.set_title('Monthly Conversion Rate by Partner', fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Month', fontsize=10)
ax.set_ylabel('Conversion Rate', fontsize=10)
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(fontsize=9, frameon=False)

plt.tight_layout()

chart_path = os.path.join(PROJECT_ROOT, 'reports/validation_charts/conversion_trend.png')
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"\nChart saved to {chart_path}")

In [ ]:
# Identify the partner with the largest conversion decline
partner_changes = {}
for partner in ['ShopBase', 'MarketHub', 'RetailCloud']:
    partner_data = monthly_conv[monthly_conv['partner'] == partner].reset_index(drop=True)
    early = partner_data[partner_data['month'].isin(early_months)]['conversion_rate'].mean()
    late  = partner_data[partner_data['month'].isin(late_months)]['conversion_rate'].mean()
    partner_changes[partner] = late - early

problem_partner = min(partner_changes, key=partner_changes.get)
print(f"Partner with largest conversion decline: {problem_partner}")
print(f"Change: {partner_changes[problem_partner]:+.1%}\n")

# Segment breakdown for problem partner
seg_conv = (
    leads_df
    .groupby(['month', 'partner', 'merchant_segment'])
    .apply(
        lambda x: (x['application_status'] == 'approved').sum() / len(x),
        include_groups=False
    )
    .reset_index()
)
seg_conv.columns = ['month', 'partner', 'merchant_segment', 'conversion_rate']

problem_seg = seg_conv[seg_conv['partner'] == problem_partner]

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

seg_colors = {
    'micro':  '#e74c3c',
    'small':  '#f39c12',
    'medium': '#2ecc71',
}

for ax, segment in zip(axes, ['micro', 'small', 'medium']):
    data = problem_seg[problem_seg['merchant_segment'] == segment]
    ax.plot(data['month'], data['conversion_rate'],
            color=seg_colors[segment], linewidth=2, marker='o', markersize=4)
    ax.set_title(f'{segment.capitalize()} merchants', fontsize=11, fontweight='bold')
    ax.set_xlabel('Month', fontsize=9)
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

axes[0].set_ylabel('Conversion Rate', fontsize=9)
fig.suptitle(f'{problem_partner} Conversion Rate by Merchant Segment',
             fontsize=13, fontweight='bold')

plt.tight_layout()

chart_path = os.path.join(PROJECT_ROOT, 'reports/validation_charts/problem_partner_segments.png')
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"{problem_partner} conversion by segment: first 6 months vs last 6 months")
print("-" * 55)
for segment in ['micro', 'small', 'medium']:
    data = problem_seg[problem_seg['merchant_segment'] == segment].reset_index(drop=True)
    early = data[data['month'].isin(early_months)]['conversion_rate'].mean()
    late  = data[data['month'].isin(late_months)]['conversion_rate'].mean()
    change = late - early
    print(f"  {segment:<8}  early: {early:.1%}  late: {late:.1%}  change: {change:+.1%}")

In [ ]:
# Build model on problem partner leads only
problem_leads = leads_df[leads_df['partner'] == problem_partner].copy()
problem_leads['approved']    = (problem_leads['application_status'] == 'approved').astype(int)
problem_leads['month_index'] = pd.to_datetime(problem_leads['date']).dt.to_period('M').apply(
    lambda x: (x.year - 2023) * 12 + x.month - 1
)

le = LabelEncoder()
problem_leads['segment_encoded'] = le.fit_transform(problem_leads['merchant_segment'])

features = ['month_index', 'segment_encoded']
X = problem_leads[features]
y = problem_leads['approved']

model = LogisticRegression(class_weight='balanced')
model.fit(X, y)

print(f"Logistic Regression: {problem_partner} approval prediction")
print("-" * 50)
print(f"\nModel accuracy: {model.score(X, y):.1%}")
print(f"\nFeature coefficients:")
for feat, coef in zip(features, model.coef_[0]):
    direction = 'decreases' if coef < 0 else 'increases'
    print(f"  {feat:<20} {coef:+.4f}  ({direction} approval odds)")

print(f"\nClassification report:")
print(classification_report(y, model.predict(X),
      target_names=['Declined/Withdrawn', 'Approved']))

In [ ]:
problem_leads_all = leads_df[leads_df['partner'] == problem_partner].copy()
problem_leads_all['month'] = pd.to_datetime(
    problem_leads_all['date']).dt.to_period('M').astype(str)

# Baseline: average conversion across first 6 months
baseline_conv = (
    problem_leads_all[problem_leads_all['month'].isin(early_months)]
    .pipe(lambda x: (x['application_status'] == 'approved').sum() / len(x))
)

# Actual conversion across last 6 months
late_leads   = problem_leads_all[problem_leads_all['month'].isin(late_months)]
actual_conv  = (late_leads['application_status'] == 'approved').sum() / len(late_leads)

# Average deal size for problem partner in last 6 months
problem_funded = funded_df[funded_df['partner'] == problem_partner].copy()
problem_funded['month'] = pd.to_datetime(
    problem_funded['funded_date']).dt.to_period('M').astype(str)
avg_deal_size = problem_funded[
    problem_funded['month'].isin(late_months)]['funded_amount'].mean()

n_late_leads       = len(late_leads)
actual_approvals   = int(n_late_leads * actual_conv)
baseline_approvals = int(n_late_leads * baseline_conv)
lost_deals         = baseline_approvals - actual_approvals
revenue_impact     = lost_deals * avg_deal_size

print(f"{problem_partner} Conversion: Revenue Impact")
print("=" * 45)
print(f"\n  Baseline conversion (months 1-6):  {baseline_conv:.1%}")
print(f"  Actual conversion (months 13-18):  {actual_conv:.1%}")
print(f"  Conversion gap:                    {actual_conv - baseline_conv:+.1%}")
print(f"\n  Leads in last 6 months:            {n_late_leads:,}")
print(f"  Actual approvals:                  {actual_approvals:,}")
print(f"  Approvals at baseline conversion:  {baseline_approvals:,}")
print(f"  Lost deals:                        {lost_deals:,}")
print(f"\n  Average deal size ({problem_partner}):  £{avg_deal_size:,.0f}")
print(f"  Estimated revenue impact:          £{revenue_impact:,.0f}")

## MarketHub Conversion Investigation
**Period:** January 2023 - June 2024

---

## Finding

MarketHub is the only partner showing a meaningful conversion decline. Its approval rate
fell from 35.9% in the first six months of the period to 31.2% in the last six, a drop
of 4.7 percentage points. ShopBase and RetailCloud were broadly flat over the same
period, up 0.5% and 1.3% respectively. The problem is isolated to one channel.

The decline affected all three merchant segments. Micro merchants were hit hardest,
with their approval rate falling to around 15% by June 2024. Small and medium merchants
also declined, though less sharply.

A logistic regression confirms the decline is time-driven. Controlling for merchant
segment, each passing month reduces the odds of a MarketHub application getting approved.
This rules out a simple explanation like a shift toward riskier merchant types applying.
Something changed in how MarketHub applications are being assessed.

The revenue cost of the conversion decline is estimated at £672k in funded volume over
the last six months, based on 47 deals that would have been approved had conversion
stayed at its baseline rate.

---

## Three hypotheses

**1. Risk criteria were tightened**

The most straightforward explanation is that underwriting standards on MarketHub were
raised at some point in the past six months. A deliberate decision to approve fewer
deals would produce exactly this pattern across all segments. If this is the case, the
decline is intentional and the £672k volume impact is the cost of a risk decision.
The question then becomes whether that trade-off was made explicitly and whether it
is still the right call.

**2. Merchant quality declined**

MarketHub merchants may have become riskier on average, causing more applications to
fail existing criteria without any change to the criteria themselves. If the platform
is attracting smaller or less established businesses over time, or if trading conditions
for MarketHub sellers have deteriorated, this would show up as a conversion decline
across all segments. Default rate trends on MarketHub would help confirm or rule this
out.

**3. A process or data issue**

Applications may be failing for operational reasons rather than credit reasons. A change
in the data MarketHub passes through at application, a system integration issue, or an
inconsistency in how applications are being processed could cause declines to rise
without any underlying change in merchant quality or risk appetite. This hypothesis is
worth ruling out quickly as it would be the easiest to fix.

---

## Recommended next step

Pull the decline and withdrawal reasons from the MarketHub application data for the
last six months and compare them to the previous period. If the split between credit
declines and process declines has shifted, that points to hypothesis three. If credit
declines are up, compare the financial profiles of declined merchants now versus a year
ago to test whether merchant quality has changed. That analysis will close off two of
the three hypotheses and give the team a clear direction.

---

## Numbers at a glance

| Metric | Months 1-6 | Months 13-18 | Change |
|---|---|---|---|
| Conversion rate | 35.9% | 31.2% | -4.7pp |
| ShopBase (reference) | 35.1% | 35.6% | +0.5pp |
| RetailCloud (reference) | 33.9% | 35.2% | +1.3pp |
| Est. lost deals | | 47 | |
| Est. revenue impact | | £672k | |

In [ ]:
os.chdir(PROJECT_ROOT)

!git add .
!git commit -m "feat: phase 3 conversion investigation, charts, model and brief"
!git push

print("Done.")

## Phase 4: ShopBase Pricing Model

### What we are doing and why

The pricing team wants to know whether reducing the factor rate on ShopBase by 0.5% is worth it. On the surface this sounds simple, but it involves a genuine trade-off. Cutting the rate means earning less on every deal. The only way it pays off is if the lower rate attracts enough extra volume to compensate for the margin you gave up.

This section builds a model to answer that question properly. We start by laying out what ShopBase currently looks like in terms of revenue and returns. We then estimate how sensitive merchants are to price changes. Finally we run three scenarios -- conservative, base, and optimistic -- to show the range of outcomes and identify the conditions under which the rate cut becomes worthwhile.

The output is a clear recommendation with the assumptions stated explicitly so the Head of Commercial can challenge them if they disagree.

Should we reduce the ShopBase factor rate by 0.5%?
This model works out the volume uplift needed to justify the margin reduction,
then tests three scenarios to see whether that hurdle is realistic.

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import os

PROJECT_ROOT = '/content/drive/MyDrive/capflow-analytics'
db_path      = os.path.join(PROJECT_ROOT, 'data/capflow.db')

conn        = sqlite3.connect(db_path)
funded_df   = pd.read_sql('SELECT * FROM funded_deals', conn)
defaults_df = pd.read_sql('SELECT * FROM defaults', conn)
conn.close()

# ShopBase deals only
sb = funded_df[funded_df['partner'] == 'ShopBase'].copy()

# Revenue = funded amount x (factor rate - 1), i.e. the fee charged
sb['revenue'] = sb['funded_amount'] * (sb['factor_rate'] - 1)

# Join defaults
sb = sb.merge(
    defaults_df[['deal_id', 'amount_outstanding']],
    on='deal_id', how='left'
)
sb['amount_outstanding'] = sb['amount_outstanding'].fillna(0)
sb['defaulted']          = (sb['amount_outstanding'] > 0).astype(int)

# Current state metrics
total_funded    = sb['funded_amount'].sum()
total_revenue   = sb['revenue'].sum()
total_defaults  = sb['amount_outstanding'].sum()
default_rate    = sb['defaulted'].mean()
avg_factor_rate = sb['factor_rate'].mean()
avg_deal_size   = sb['funded_amount'].mean()
n_deals         = len(sb)

risk_adj_return = (total_revenue - total_defaults) / total_funded

print("ShopBase: Current State")
print("=" * 45)
print(f"\n  Total deals funded:        {n_deals:,}")
print(f"  Total funded volume:       £{total_funded:,.0f}")
print(f"  Average deal size:         £{avg_deal_size:,.0f}")
print(f"  Average factor rate:       {avg_factor_rate:.3f}")
print(f"\n  Gross revenue (fees):      £{total_revenue:,.0f}")
print(f"  Default losses:            £{total_defaults:,.0f}")
print(f"  Net revenue:               £{total_revenue - total_defaults:,.0f}")
print(f"\n  Default rate:              {default_rate:.1%}")
print(f"  Risk-adjusted return:      {risk_adj_return:.1%}")

In [ ]:
# How much extra volume is needed to offset the margin reduction?

# Current revenue per £1 funded = factor_rate - 1
# After rate cut: new_rate = factor_rate - 0.005
# Revenue lost per deal from rate cut
rate_cut        = 0.005
new_factor_rate = avg_factor_rate - rate_cut

current_margin  = avg_factor_rate - 1
new_margin      = new_factor_rate - 1
margin_reduction_pct = rate_cut / current_margin

print("Breakeven Analysis")
print("=" * 45)
print(f"\n  Current avg factor rate:   {avg_factor_rate:.3f}")
print(f"  New factor rate:           {new_factor_rate:.3f}")
print(f"  Rate cut:                  {rate_cut:.3f}")
print(f"\n  Current margin per £1:     {current_margin:.3f}")
print(f"  New margin per £1:         {new_margin:.3f}")
print(f"  Margin reduction:          {margin_reduction_pct:.1%}")

# Revenue lost per deal at current average deal size
revenue_lost_per_deal = avg_deal_size * rate_cut
print(f"\n  Revenue lost per deal:     £{revenue_lost_per_deal:,.0f}")

# Total revenue lost on existing book if rate cut applied immediately
total_revenue_lost = total_funded * rate_cut
print(f"  Total revenue lost on      ")
print(f"  existing book:             £{total_revenue_lost:,.0f}")

# Extra deals needed to break even
# Each new deal generates: avg_deal_size * new_margin
revenue_per_new_deal = avg_deal_size * new_margin
extra_deals_needed   = total_revenue_lost / revenue_per_new_deal
breakeven_volume_pct = extra_deals_needed / n_deals

print(f"\n  Revenue per new deal:      £{revenue_per_new_deal:,.0f}")
print(f"  Extra deals needed to      ")
print(f"  break even:                {extra_deals_needed:.0f} deals")
print(f"  As % of current book:      {breakeven_volume_pct:.1%}")
print(f"\n  Interpretation: the rate cut breaks even if it drives")
print(f"  at least {breakeven_volume_pct:.1%} more ShopBase deals.")

In [ ]:
# Price elasticity assumption
# How much does conversion improve for every 1% reduction in factor rate?
# We do not have enough historical rate variation to estimate this precisely,
# so we use a range of assumptions.

# Elasticity = % change in conversion / % change in factor rate
# Conservative: low elasticity (merchants are not very price sensitive)
# Base: moderate elasticity
# Optimistic: high elasticity

scenarios = {
    'Conservative': {'volume_uplift': 0.10, 'color': '#e74c3c'},
    'Base':         {'volume_uplift': 0.20, 'color': '#f39c12'},
    'Optimistic':   {'volume_uplift': 0.35, 'color': '#2ecc71'},
}

print("Scenario Analysis: 0.5% Factor Rate Reduction on ShopBase")
print("=" * 60)
print(f"\n{'Scenario':<14} {'Vol Uplift':>10} {'New Deals':>10} "
      f"{'New Volume':>14} {'Rev Lost':>12} {'Rev Gained':>12} {'Net':>12}")
print("-" * 60)

scenario_results = {}

for scenario, params in scenarios.items():
    uplift      = params['volume_uplift']
    new_deals   = int(n_deals * uplift)
    new_volume  = new_deals * avg_deal_size

    # Revenue lost on existing book from margin reduction
    rev_lost    = total_funded * rate_cut

    # Revenue gained from new deals at new margin
    rev_gained  = new_volume * new_margin

    # Net revenue impact
    net_revenue = rev_gained - rev_lost

    # NPV positive if net_revenue > 0
    npv_positive = "YES" if net_revenue > 0 else "NO"

    scenario_results[scenario] = {
        'uplift':      uplift,
        'new_deals':   new_deals,
        'new_volume':  new_volume,
        'rev_lost':    rev_lost,
        'rev_gained':  rev_gained,
        'net_revenue': net_revenue,
        'npv_positive': npv_positive,
    }

    print(f"{scenario:<14} {uplift:>9.0%} {new_deals:>10,} "
          f"£{new_volume:>12,.0f} £{rev_lost:>10,.0f} "
          f"£{rev_gained:>10,.0f} £{net_revenue:>10,.0f}  {npv_positive}")

print("-" * 60)
print(f"\n  Breakeven volume uplift: {breakeven_volume_pct:.1%}")
print(f"\n  Conservative ({scenarios['Conservative']['volume_uplift']:.0%} uplift): "
      f"{'above' if scenario_results['Conservative']['net_revenue'] > 0 else 'below'} breakeven")
print(f"  Base         ({scenarios['Base']['volume_uplift']:.0%} uplift): "
      f"{'above' if scenario_results['Base']['net_revenue'] > 0 else 'below'} breakeven")
print(f"  Optimistic   ({scenarios['Optimistic']['volume_uplift']:.0%} uplift): "
      f"{'above' if scenario_results['Optimistic']['net_revenue'] > 0 else 'below'} breakeven")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Chart 1: Net revenue by scenario
scenario_names = list(scenario_results.keys())
net_revenues   = [scenario_results[s]['net_revenue'] for s in scenario_names]
colors         = [scenarios[s]['color'] for s in scenario_names]

bars = axes[0].bar(scenario_names, net_revenues, color=colors,
                   width=0.5, edgecolor='white', linewidth=0.8)

axes[0].axhline(y=0, color='#2c3e50', linewidth=1.2, linestyle='--')
axes[0].set_title('Net Revenue Impact by Scenario', fontsize=11,
                  fontweight='bold', pad=12)
axes[0].set_ylabel('Net Revenue Impact (£)', fontsize=9)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

for bar, val in zip(bars, net_revenues):
    label_y = bar.get_height() + abs(max(net_revenues)) * 0.03 if val >= 0 \
              else bar.get_height() - abs(max(net_revenues)) * 0.06
    axes[0].text(bar.get_x() + bar.get_width() / 2, label_y,
                 f'£{val:,.0f}', ha='center', va='bottom',
                 fontsize=9, fontweight='bold', color='#2c3e50')

# Chart 2: Revenue lost vs gained by scenario
x     = np.arange(len(scenario_names))
width = 0.35

rev_lost_vals   = [scenario_results[s]['rev_lost']   for s in scenario_names]
rev_gained_vals = [scenario_results[s]['rev_gained'] for s in scenario_names]

axes[1].bar(x - width/2, rev_lost_vals,   width, label='Revenue lost (margin)',
            color='#e74c3c', edgecolor='white')
axes[1].bar(x + width/2, rev_gained_vals, width, label='Revenue gained (volume)',
            color='#2ecc71', edgecolor='white')

axes[1].set_title('Revenue Lost vs Gained by Scenario', fontsize=11,
                  fontweight='bold', pad=12)
axes[1].set_ylabel('£', fontsize=9)
axes[1].set_xticks(x)
axes[1].set_xticklabels(scenario_names, fontsize=9)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)
axes[1].legend(fontsize=8, frameon=False)

plt.suptitle('ShopBase Factor Rate Reduction: 0.5% Scenario Analysis',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()

chart_path = os.path.join(PROJECT_ROOT, 'reports/validation_charts/shopbase_pricing.png')
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Chart saved to {chart_path}")

## ShopBase Pricing: Factor Rate Reduction Analysis

---

## The question

Should CapFlow reduce the ShopBase factor rate by 0.5%? The rate cut would mean earning
less on every deal. The question is whether the lower rate attracts enough extra volume
to more than compensate for that margin reduction.

---

## Current position

ShopBase has funded 1,433 deals totalling £21.4M over the 18-month period. The average
factor rate is 1.237, generating £4.5M in gross fee income. After default losses of
£696k, net revenue is £3.8M, giving a risk-adjusted return of 17.6% on funded volume.
The default rate of 7.1% is the highest across the three partners, driven primarily by
micro-merchant defaults.

---

## What the model shows

The rate cut needs to drive just 2.2% more deals to break even. That is 31 additional
deals on the current book. The reason the hurdle is low is that the margin reduction in
pounds is relatively small at £107k, while each new deal generates around £3,500 in
revenue at the new rate.

We tested three volume uplift scenarios:

| Scenario | Volume uplift assumed | Net revenue impact |
|---|---|---|
| Conservative | 10% (143 new deals) | +£389k |
| Base | 20% (286 new deals) | +£885k |
| Optimistic | 35% (501 new deals) | +£1.6M |

All three scenarios are above breakeven. The rate cut is NPV-positive under any
reasonable assumption about volume response.

---

## The assumptions that drive this

The model rests on one assumption that cannot be verified from historical data alone:
that a 0.5% rate reduction actually drives meaningful volume uplift. We do not have
enough historical rate variation on ShopBase to estimate price elasticity with
confidence. The 10%, 20%, and 35% uplift figures are assumptions, not measurements.

Two things should be true for the conservative scenario to hold:
- ShopBase merchants are at least moderately price-sensitive
- CapFlow's rate is competitive enough that a small reduction changes purchasing decisions

If merchants are largely indifferent to a 0.5% rate movement, the volume uplift could
be close to zero, in which case the business loses £107k with nothing in return.

---

## Recommendation

Do it, but run it as a controlled test first. Apply the rate reduction to a subset of
ShopBase merchants for 60 to 90 days and measure whether conversion and deal volume
actually respond. The cost of the test is small -- £107k in margin on the full book,
less on a subset -- and it will produce a real elasticity estimate that makes future
pricing decisions far more grounded.

If the test shows even a 3% to 5% volume response, the full rollout is clearly
justified. If it shows no response, the rate cut does not make sense and you have
learned something valuable about ShopBase merchant price sensitivity at low cost.

---

## Key assumptions

- Volume uplift figures are assumptions based on reasonable commercial judgement,
  not measured elasticity
- Default rate is held constant across scenarios. A lower rate may attract riskier
  merchants, which would increase defaults and reduce the net benefit
- Analysis covers the full 18-month book. Forward-looking impact depends on
  deal volumes in the next period